# Morning class 27/08 — Extra practice 02 SOLUTIONS: for loops   (L05)

Every cell below was executed on the same Python the lab ships (3.13), and the
quoted output is what it actually printed.

Question 4 drops a data point without saying so. That is the one to re-read.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Extra practice 02 — For loops. Run this once.
cities = ["Toronto", "Lisbon", "Osaka", "Nairobi"]
populations = [2794, 545, 2691]          # thousands -- note the length
temps = [7, 17, 16, 26]

header = "order_id,customer,amount,status"
line = "1042,ana,88.50,shipped"

stock = {"widget": 12, "gizmo": 4, "sprocket": 30}

print(len(cities), "cities,", len(populations), "populations,",
      len(temps), "temperatures")

### Question 1

Names and numbering. -> `Toronto 7`, `Lisbon 6`, `Osaka 5`, `Nairobi 7`; then `1. Toronto` through `4. Nairobi`.

`enumerate(cities, start=1)` numbers from 1 without any `i + 1` arithmetic
in the body. The deck does not mention `start`, and it is the difference
between a report that reads `1.` and one that reads `0.`.

The f-string `f"{i}. {city}"` puts the pieces together without the spaces
that `print(i, ".", city)` would insert.

In [ ]:
for city in cities:
    print(city, len(city))

print("---")

for i, city in enumerate(cities, start=1):
    print(f"{i}. {city}")

### Question 2

Zipping two lists. -> `Toronto: 7`, `Lisbon: 17`, `Osaka: 16`, `Nairobi: 26`, then `<zip object at 0x…>`.

`zip` walks two collections in step and hands you a tuple per position,
which the loop header unpacks into two names.

Like `enumerate`, it is **lazy** — printing it shows an object, not the
pairs, and it can only be walked once. `list(zip(...))` materialises it if
you need to look twice. The hex address is a memory location and means
nothing.

In [ ]:
for city, temp in zip(cities, temps):
    print(f"{city}: {temp}")

print(zip(cities, temps))

### Question 3

Sorting in the loop header. -> alphabetical `Lisbon, Nairobi, Osaka, Toronto`; then by length `Toronto 7, Nairobi 7, Lisbon 6, Osaka 5`; then `cities` unchanged in its original order.

`sorted()` returns a **new** list and leaves the original alone — which is
why the third print still shows the input order. `cities.sort()` would
have rearranged `cities` in place and returned `None`, so
`cities = cities.sort()` destroys the list. That is worth knowing before
you do it.

`key=len` sorts by a computed value rather than the item itself. Toronto
and Nairobi are both 7 and Toronto comes first because it came first in the
input — Python's sort is *stable*, meaning ties keep their original order.
That is a guarantee, not luck, and it is what lets you sort by one field
and then another.

In [ ]:
for city in sorted(cities):
    print(city)

print("---")

for city in sorted(cities, key=len, reverse=True):
    print(city, len(city))

print("---")
print(cities)

### Question 4

Zipping lists of different lengths. -> three pairs, then `3 pairs from 4 cities and 3 populations`, and `dropped: ['Nairobi']`.

`zip` stops at the **shortest** input. Nairobi has no population, so the
pair was never made — no error, no warning, no mention. Four cities went
in and three came out.

This is the silent data loss to watch for. Every time you zip two lists
that came from different places, one of them can be short: a filter applied
to one and not the other, a header row skipped once, an API page that
ended early. The result is a report that is quietly missing its last row.

**Check the lengths before you zip**, or use
`itertools.zip_longest` if a short list is expected and you want padding
rather than truncation.

In [ ]:
pairs = 0
for city, pop in zip(cities, populations):
    print(city, pop)
    pairs = pairs + 1

print(pairs, "pairs from", len(cities), "cities and",
      len(populations), "populations")

# zip stops at the SHORTEST input. Nairobi has no population, so the pair
# was never made and never mentioned.
print("dropped:", cities[len(populations):])

### Question 5

Parsing a CSV line. -> four `column = value` lines, then `{'order_id': '1042', 'customer': 'ana', 'amount': '88.50', 'status': 'shipped'}`.

`header.split(",")` and `line.split(",")` give two lists in matching order,
and `zip` puts them back together. That is the whole idea behind reading a
CSV by hand, and it is what `csv.DictReader` does for you.

Every value in `record` is a **string**, including `'1042'` and `'88.50'`.
Nothing has been converted, and `record["amount"] > 50` would raise exactly
as worksheet 01 Q11 did. Convert at the boundary.

And this parser breaks on the first address field containing a comma, which
is why the real `csv` module exists.

In [ ]:
columns = header.split(",")
values = line.split(",")

for column, value in zip(columns, values):
    print(column, "=", value)

record = {}
for column, value in zip(columns, values):
    record[column] = value

print(record)

### Question 6

Stock levels. -> `widget 12 -- ok`, `gizmo 4 -- low`, `sprocket 30 -- ok`, then `1 low of 3 products; 46 units total`.

One pass doing three jobs: printing a label, counting the low ones,
totalling the units. The counter and the accumulator are both initialised
before the loop and read after it.

`.items()` with two loop variables is the form to default to when you need
both the key and the value. `for name in stock:` followed by `stock[name]`
works but looks the value up a second time for no reason.

In [ ]:
low = 0
units = 0

for name, count in stock.items():
    units = units + count
    if count < 10:
        print(name, count, "-- low")
        low = low + 1
    else:
        print(name, count, "-- ok")

print(low, "low of", len(stock), "products;", units, "units total")

### Question 7

A list of lists. -> `[['Toronto', 7], ['Lisbon', 17], ['Osaka', 16], ['Nairobi', 26]]`, then four `X is N degrees` lines.

`range(len(cities))` is used here because two lists have to be indexed in
step — the one case where the index really is needed. `zip(cities, temps)`
would be better and is what Q2 already did.

The second loop unpacks each two-item row into two names, exactly as it
would for a tuple. Unpacking works on any sequence of the right length,
not only tuples.

In [ ]:
table = []
for i in range(len(cities)):
    table.append([cities[i], temps[i]])

print(table)

for city, temp in table:
    print(f"{city} is {temp} degrees")

### Question 8

Counting vowels. -> `Nairobi 4`; then `Toronto 3`, `Lisbon 2`, `Osaka 3`, `Nairobi 4`.

`char.lower() in "aeiou"` uses `in` on a **string**, which is a substring
test — for a single character that is exactly a membership test, and it
saves writing five `or`s.

The second half is a nested loop, and the `count = 0` sits at the top of
the **outer** body so it resets for each city. Put it above the outer loop
instead and you get a running total across all four — no error, four
increasing numbers, and it looks almost right.

In [ ]:
count = 0
for char in "Nairobi":
    if char.lower() in "aeiou":
        count = count + 1
print("Nairobi", count)

print("---")

for city in cities:
    count = 0
    for char in city:
        if char.lower() in "aeiou":
            count = count + 1
    print(city, count)

### Question 9

Indexing two lists of different lengths. -> three lines, then `IndexError: list index out of range`.

`range(len(cities))` counts 0, 1, 2, 3 — but `populations` stops at index
2. The fourth pass raises.

Put this next to Q4, which did the same job with `zip` and lost Nairobi in
silence. **Same mismatch, two opposite failure modes:** `zip` truncates
quietly, indexing raises loudly.

Which one you want is a real decision. If a missing population is expected,
`zip` is right. If it means the data is broken, the `IndexError` is the
kinder outcome — it stops you shipping a report that is short a city.

In [ ]:
for i in range(len(cities)):
    print(cities[i], populations[i])

# This is SUPPOSED to raise: IndexError, on the fourth pass. There are four
# cities and three populations, and range(len(cities)) does not know that.
#
# Q4's zip() is the safe version -- it stops at the shorter list. Note the
# difference in failure mode: zip loses a city SILENTLY, this raises. Which
# one you want depends on whether a missing population is normal or a bug.